
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D19448A660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D19448B0E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### Lets build a recipe based system

In [2]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    name:str=Field(description="The name of the dish")
    cuisine:str=Field(description="The cuisine style of the dish")
    prep_time:int=Field(description="Preparation time in minutes")
    ingredients:list[str]=Field(description="List of ingredients needed for the dish")

In [ ]:
# not an model invocation, with_structured_output(Recipe) just a runnable
model_with_structure=model.with_structured_output(Recipe)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D196F67110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D196F67B10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Recipe', 'description': 'A recipe with details.', 'parameters': 

In [4]:
model.invoke("Tell me about khachapuri, the Georgian cheese-filled bread")

AIMessage(content='### Khachapuri: Georgia’s Iconic Cheese‑Filled Bread\n\n**Khachapuri (ხაჭაპური)** is more than just a dish—it’s a symbol of Georgian hospitality, a culinary ambassador, and a staple that can be found in almost every Georgian home, café, or restaurant around the world. The name itself comes from the Georgian words **“khach”** (bread) and **“apuri”** (to bake), literally “to bake bread.” But what makes khachapuri truly special is the way it marries a golden, yeasted crust with a molten, cheesy center that can be served in a variety of creative shapes and flavors.\n\n---\n\n## 1. The Core Ingredients\n\n| Ingredient | Typical Use | Notes |\n|------------|-------------|-------|\n| **Flour** | 500\u202fg (about 4 cups) | All‑purpose or bread flour for a chewy crumb |\n| **Water** | 250\u202fml (1\u202f½ cups) | Warm (about 40\u202f°C / 104\u202f°F) |\n| **Yeast** | 7\u202fg (1 packet) | Active dry or instant |\n| **Salt** | 10\u202fg (2\u202ftsp) | |\n| **Sugar** | 10\u20

In [5]:
response=model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")
response

Recipe(name='Khachapuri', cuisine='Georgian', prep_time=90, ingredients=['1 1/2 cups warm water', '1 tbsp sugar', '1 tsp active dry yeast', '3 1/2 cups all-purpose flour', '1 tsp salt', '1 tbsp olive oil', '1 cup shredded mozzarella cheese', '1 cup shredded feta cheese', '1 egg, beaten', '1 tsp butter, melted'])

### MEssage output alongside parsed structure

In [6]:
from pydantic import BaseModel, Field


class Recipe(BaseModel):
    """A recipe with details."""
    name: str = Field(..., description="The name of the dish")
    cuisine: str = Field(..., description="The cuisine style of the dish")
    prep_time: int = Field(..., description="Preparation time in minutes")
    ingredients: list[str] = Field(..., description="List of ingredients needed for the dish")

model_with_structure = model.with_structured_output(Recipe, include_raw=True)  

response = model_with_structure.invoke("Provide the recipe details for khachapuri, the Georgian cheese-filled bread")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide recipe details for khachapuri. The user wants "Provide the recipe details". We have a function to call: functions.Recipe with arguments: cuisine, ingredients, name, prep_time. We should call the function. The name: "Khachapuri". Cuisine: "Georgian". Ingredients list: typical ingredients: flour, yeast, salt, sugar, water, butter, cheese (a mix of sulguni, feta, mozzarella). Also optional eggs, milk, etc. Prep time: maybe 60 minutes.\n\nWe should call the function.', 'tool_calls': [{'id': 'fc_86f252c8-a6ed-4f5e-8e16-be3f5fd64219', 'function': {'arguments': '{"cuisine":"Georgian","ingredients":["2 cups all-purpose flour","1 packet active dry yeast (2 ¼ tsp)","1 tsp sugar","1 tsp salt","¾ cup warm water","2 tbsp melted butter","1 cup shredded sulguni cheese (or a mix of sulguni, feta, mozzarella)","1 egg (for egg wash)"],"name":"Khachapuri","prep_time":60}', 'name': 'Recipe'}, 'type': 'function'}]}, r

### Nested Structure

In [7]:
from pydantic import BaseModel, Field


class Ingredient(BaseModel):
    item: str
    quantity: str

class RecipeDetails(BaseModel):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for khachapuri, the Georgian cheese-filled bread")
response

RecipeDetails(name='Khachapuri', cuisine='Georgian', ingredients=[Ingredient(item='All-purpose flour', quantity='4 cups (480 g)'), Ingredient(item='Active dry yeast', quantity='2 ¼ teaspoons (7 g)'), Ingredient(item='Warm milk', quantity='1 cup (240 ml)'), Ingredient(item='Salt', quantity='1 ½ teaspoons (8 g)'), Ingredient(item='Sugar', quantity='1 teaspoon (4 g)'), Ingredient(item='Vegetable oil', quantity='2 tablespoons (30 ml)'), Ingredient(item='Butter', quantity='1 tablespoon (14 g), melted'), Ingredient(item='Eggs', quantity='2 (for dough) + 1 (for egg wash)'), Ingredient(item='Sulguni cheese', quantity='1 cup (120 g) shredded'), Ingredient(item='Feta cheese', quantity='1 cup (120 g) crumbled'), Ingredient(item='Mozzarella cheese', quantity='½ cup (60 g) shredded'), Ingredient(item='Freshly cracked black pepper', quantity='to taste'), Ingredient(item='Optional: Butter for topping', quantity='2 tablespoons (28 g)')], steps=['1. In a small bowl, dissolve the yeast and sugar in the 

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [9]:
from typing import Annotated

from typing_extensions import TypedDict


class RecipeDict(TypedDict):
    """A recipe with details."""
    name: Annotated[str, ..., "The name of the dish"]
    cuisine: Annotated[str, ..., "The cuisine style of the dish"]
    prep_time: Annotated[int, ..., "Preparation time in minutes"]
    ingredients: Annotated[list[str], ..., "List of ingredients needed for the dish"]


model_withtypedict=model.with_structured_output(RecipeDict)
response=model_withtypedict.invoke("Please provide the recipe details for pizza")
response

{'cuisine': 'Italian',
 'ingredients': ['Pizza dough',
  'Tomato sauce',
  'Mozzarella cheese',
  'Fresh basil leaves',
  'Olive oil',
  'Salt',
  'Pepper'],
 'name': 'Classic Margherita Pizza',
 'prep_time': 30}

In [15]:
bad = RecipeDict(
    name=123,                 # should be str
    cuisine=None,             # should be str
    prep_time="not a number", # should be int
    ingredients="cheese",     # should be list[str]
)
print(bad)   # no error

{'name': 123, 'cuisine': None, 'prep_time': 'not a number', 'ingredients': 'cheese'}


In [13]:
class Ingredient(TypedDict):  # noqa: F811
    item: str
    quantity: str

class RecipeDetails(TypedDict):
    name: str
    cuisine: str
    ingredients: list[Ingredient]
    steps: list[str]
    prep_time: float | None = Field(None, description="Prep time in minutes")

model_with_structure = model.with_structured_output(RecipeDetails)

response = model_with_structure.invoke("Provide the full recipe for pancakes")
response

{'cuisine': 'American',
 'ingredients': [{'item': 'All-purpose flour', 'quantity': '1 1/2 cups'},
  {'item': 'Baking powder', 'quantity': '3 1/2 teaspoons'},
  {'item': 'Salt', 'quantity': '1 teaspoon'},
  {'item': 'White sugar', 'quantity': '1 tablespoon'},
  {'item': 'Milk', 'quantity': '1 1/4 cups'},
  {'item': 'Egg', 'quantity': '1 large'},
  {'item': 'Melted butter', 'quantity': '3 tablespoons'},
  {'item': 'Vanilla extract', 'quantity': '1 teaspoon'}],
 'name': 'Classic Pancakes',
 'prep_time': 15,
 'steps': ['In a large mixing bowl, sift together the flour, baking powder, salt, and sugar.',
  'Make a well in the center and pour in the milk, egg, melted butter, and vanilla. Mix until just combined; a few lumps are fine.',
  'Heat a lightly oiled griddle or frying pan over medium-high heat.',
  'Pour or scoop the batter onto the griddle, using about 1/4 cup for each pancake. Cook until bubbles form on the surface and the edges look set, about 2-3 minutes.',
  'Flip with a spatula 

In [14]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2025-08-05',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [16]:
import os

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

In [17]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029', additional_kwargs={}, response_metadata={}, id='63d7fdf3-dcb0-4c8b-b018-158c92520add'),
  AIMessage(content='{"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}', additional_kwargs={'reasoning_content': 'We need to output JSON in the format ContactInfo. Must be compact JSON, all fields required: name, email, phone. Provide values. Ensure JSON is valid. Let\'s produce: {"name":"Elena Marquez","email":"elena.marquez@brightleaf-consulting.com","phone":"(312) 774-5029"}'}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 247, 'total_tokens': 364, 'completion_time': 0.12010421, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.01209366, 'prompt_tokens_details': None, 'queue_time': 0.366472598, 'total_time': 0.13219787}, 'model_name': 'openai/gpt-oss-20b', 'sys

In [18]:
result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

In [19]:
## Dataclass

from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: Elena Marquez, elena.marquez@brightleaf-consulting.com, (312) 774-5029"}]
})

result["structured_response"]

ContactInfo(name='Elena Marquez', email='elena.marquez@brightleaf-consulting.com', phone='(312) 774-5029')

### dataclass and pydantic in same example

In [20]:
from dataclasses import dataclass

from pydantic import BaseModel, ValidationError


@dataclass
class ContactDC:
    name: str
    email: str
    phone: str

class ContactPD(BaseModel):
    name: str
    email: str
    phone: str

print(ContactDC(name=123, email=None, phone=["x"]))   # works, no error

try:
    ContactPD(name=123, email=None, phone=["x"])
except ValidationError as e:
    print(e)                                           # raises

ContactDC(name=123, email=None, phone=['x'])
3 validation errors for ContactPD
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
email
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
phone
  Input should be a valid string [type=string_type, input_value=['x'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
